In [0]:
from datetime import date, timedelta
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
%run ../../utils/reader_utils

In [0]:

%run ../../utils/writer_utils

In [0]:
%run ../../utils/transform_utils

In [0]:
# Configs
configs = dict(dbutils.notebook.entry_point.getCurrentBindings())
today = date.today()

ENV = configs.get("env", "dev")
INITIAL_RUN = configs.get('initial_run', False)
START_DATE = configs.get("start_date") or today - timedelta(days=1)
END_DATE = configs.get("end_date") or today + timedelta(days=1)
CATALOG = f"sl_{ENV}"

print(ENV, INITIAL_RUN, START_DATE, END_DATE, CATALOG)

In [0]:
# Constants
SILVER_SHIPMENT_TABLE_CONF = {
    "table": "pyspark_shipment_events",
    "schema": "silver",
    "timestamp_col": "_insert_update_ts"
}

DIM_VEHICLE_TABLE_CONF = {
    "table": "pyspark_dim_vehicle",
    "schema": "gold",
}

DIM_DATE_TABLE_CONF = {
    "table": "pyspark_dim_date",
    "schema": "gold",
}

TARGET_TABLE_CONF = {
    "table": "pyspark_fact_shipment_event",
    "schema": "gold",
    "merge_keys": ["event_id"],
    "primary_key": "event_id"
}

In [0]:
spark.sql(f"USE CATALOG {CATALOG}")

In [0]:
silver_shipment_df = (read_delta_table(SILVER_SHIPMENT_TABLE_CONF, START_DATE, END_DATE)
                      .drop("_insert_update_ts"))

dim_vehicle_df = (spark.table(f"""{DIM_VEHICLE_TABLE_CONF.get("schema")}.
                        {DIM_VEHICLE_TABLE_CONF.get("table")}"""))
                        
dim_date_df = (spark.table(f"""{DIM_DATE_TABLE_CONF.get("schema")}.
                        {DIM_DATE_TABLE_CONF.get("table")}"""))

In [0]:
fact_shipment_event_df = (silver_shipment_df.alias("main")
                             .join(dim_vehicle_df.alias("v"), 
                                   on="vehicle_id", 
                                   how="left")
                             .join(dim_date_df.alias("d"), 
                                   on=(F.col("main.event_date") == F.col("d.full_date")), 
                                   how="left")
                             .select("main.event_id",
                                     "v.vehicle_sk",
                                     "d.date_key",
                                     *[F.col(c).alias(f"main.{c}") for c in silver_shipment_df.columns
                                       if c != "event_id"])
                              
                                     
)

In [0]:
if INITIAL_RUN: #this is not an ideal production pattern but more straight forward for this used case
    target_table = f"{TARGET_TABLE_CONF.get('schema')}.{TARGET_TABLE_CONF.get('table')}"
    primary_key = TARGET_TABLE_CONF.get('primary_key')

    (fact_shipment_event_df
     .writeTo(target_table)
     .using("delta")
     .createOrReplace())
    
    spark.sql(f"ALTER TABLE {target_table} CLUSTER BY AUTO")
    spark.sql(f"ALTER TABLE {target_table} ALTER COLUMN {primary_key} SET NOT NULL")
    spark.sql(f"ALTER TABLE {target_table} ADD CONSTRAINT {primary_key}_pk PRIMARY KEY ({primary_key})")

    spark.sql(f"""ALTER TABLE {target_table} ADD CONSTRAINT fct_shipmt_vehicle_sk_fk FOREIGN KEY (vehicle_sk) 
              REFERENCES {DIM_VEHICLE_TABLE_CONF.get('schema')}.{DIM_VEHICLE_TABLE_CONF.get('table')}(vehicle_sk)""")

    spark.sql(f"""ALTER TABLE {target_table} ADD CONSTRAINT fct_shipmt_date_key_fk FOREIGN KEY (date_key) 
              REFERENCES {DIM_DATE_TABLE_CONF.get('schema')}.{DIM_DATE_TABLE_CONF.get('table')}(date_key)""")
    
    dbutils.notebook.exit(f"Initial Table: {target_table} Setup Finished Successfully")

In [0]:
upsert_to_delta(fact_shipment_event_df, TARGET_TABLE_CONF)